# LangChain

- A tool that helps in invoking a wide variety of AI models and manage them in end - end applications. (Orchestration)
- can help connecting the agent to system DB or other data sources. (RAGs)

A framework that makes it easy to build AI Apps.

## Topic:

- Langchain basic
- Prompt templates
  - chains
  - chat roles with prompt templates
- Agent using langchain
  - tool calls
  - agentic loop
- langchain middlewares
- Structured Output
  - `with_strucutred_output` using `json_schema`, `function_calling`, `json`
  - `create_agent` using providers like `ToolProvider` etc
- Short Term Memory

## Imports and Dependencies

In [41]:
import re
import json
from pathlib import Path
from fnmatch import fnmatch
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from pydantic import BaseModel, Field, EmailStr, field_validator
from dotenv import load_dotenv
from os import getenv
from enum import Enum
from typing import List, Optional, Literal

# Lang chain
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents.middleware import AgentMiddleware, ToolCallRequest
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver  
from langgraph.graph import StateGraph

load_dotenv()
model = "openrouter/free"

llm = ChatOpenRouter(
    model_name=model,
)

## Basics:

In [2]:
prompt = "what is binary search. Write code for it."

response = llm.invoke([
    HumanMessage(content=prompt)
])

print(response.content)

### What is Binary Search?

**Binary Search** is an efficient algorithm used to find the position of a target value within a **sorted array**. 

Unlike a linear search (which checks every single element one by one), binary search uses a "divide and conquer" strategy. It repeatedly divides the search interval in half.

#### How it works:
1.  **Start** with the entire sorted array.
2.  **Find the middle element.**
3.  **Compare** the middle element with your target value:
    *   If the target **equals** the middle element, you found it! Return the index.
    *   If the target is **smaller** than the middle element, discard the right half and repeat the search on the left half.
    *   If the target is **larger** than the middle element, discard the left half and repeat the search on the right half.
4.  **Repeat** until the target is found or the search interval is empty.

#### Efficiency (Complexity):
*   **Time Complexity:** $O(\log n)$ — This is extremely fast. For example, searching 

- LangChain provides a different API than the ones present from OpenAI client.
- For Passing Human and Assistant message inbuilt functions already exist.
- The concepts for agentic loop etc are the same here as well.
- https://docs.langchain.com/oss/python/integrations/chat/openrouter - for other parameters when creating the model.
  - for tools and stuff the lang chain follows chat completions api format, check above docs
  - other paramerters like `max_tokens` also exist to limit the output tokens.

## Prompt Template and Chains:

- Reusable instructions which can be passed to langchain with different parameters.

In [3]:
sample_template = """
You are an expert CS Engineer explain the following algorithm :-
{algorithm_name}

Follow the exact structure :-

### Concept
Explain what problem this algo solves.

### Efficiency
Explain time and space complexity

### Sample Trace run
Show a simple dry run using {sample_input}

Answer::
"""

sample_prompt_template = PromptTemplate(template=sample_template, input_variables=["algorithm_name", "sample_input"])

In [4]:
prompt = sample_prompt_template.format(algorithm_name="Binary Search", sample_input="1 2 3 4 5")
print(prompt)
print("\n*****\n")
print(llm.invoke(prompt).content)
print("\n*****\n")


You are an expert CS Engineer explain the following algorithm :-
Binary Search

Follow the exact structure :-

### Concept
Explain what problem this algo solves.

### Efficiency
Explain time and space complexity

### Sample Trace run
Show a simple dry run using 1 2 3 4 5

Answer::


*****

### Concept
Binary Search is a classic algorithm for finding a target value within a **sorted** array (or list).  
The problem it solves is: *given a sorted collection of items and a key to locate, determine the index of that key (or confirm its absence) in the smallest number of comparisons possible*.  
Instead of scanning every element (linear search), binary search repeatedly halves the search interval, discarding the half that cannot contain the target. This yields logarithmic time complexity.

### Efficiency
- **Time Complexity:**  
  - Best case: O(1) – the middle element is the target on the first check.  
  - Average and worst case: O(log n) – the search space is halved each iteration, where

---
- Above was one way we could invoke our new prompt.
- However langchain allows us to chain operations like invoking llm generating prompts

---

#### Chains:

In [5]:
# prompt feeded inside the LLM
sample_chain = sample_prompt_template | llm
result = sample_chain.invoke({
    "algorithm_name":"Binary Search", 
    "sample_input":"1 2 3 4 5"
})
print(result.content)

### Concept
Binary Search is an efficient algorithm for finding a target value within a **sorted** array or list. It works by repeatedly dividing the search interval in half. If the value of the search key is less than the item in the middle of the interval, the algorithm narrows the search to the lower half. Otherwise, it narrows the search to the upper half. This process continues until the target is found or the interval is empty. It is significantly faster than linear search for large datasets.

### Efficiency
*   **Time Complexity:** $O(\log n)$. In the worst case, the algorithm performs a comparison with half of the remaining elements at each step. This results in a logarithmic number of steps relative to the input size $n$.
*   **Space Complexity:** $O(1)$ for the iterative implementation (using constant extra space for pointers). If implemented recursively, the space complexity is $O(\log n)$ due to the recursion stack depth.

### Sample Trace run
Consider a sorted array `[1, 2

---
- Above we created a simple chain and passed our parameter directly in chain in a dictionary which would be fed to prompt template which is then fed into llm
---

#### Chat Roles:

In [6]:
sample_template_2 = ChatPromptTemplate.from_messages([
    {"role": "system", "content":"some system prompt"},
    {"role":"human", "content": "some human prompt"},
    {"role": "ai","content": "some ai response"}
])

- Now we can create chain with this and invoke the llm.
- Everything remains the same as before.

## Agents in Lang Chain:
- Previously we manually handled writing agentic loop
  - handling tool call
  - passing messages
  - max turn tracking etc
- Lang chain simplifies this process

https://docs.langchain.com/oss/python/langchain/agents

- The `create_agent` allows you to create a complete harness not just an agentic loop.
  - it automatically converts the messages into appropriate messages with roles.
  - you can pass tools information.
    - It automatically handles tool schema definition.
  - mcp information.
  - skills.
  - memory.
  - middlewares.

In [7]:
@tool
def get_weather(location: str) -> str:
    """
    This returns the weather of the provided location
    """
    return "The weather is 20 Celcius"

In [8]:
llm = ChatOpenRouter(
    model_name=model,
)

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a weather agent. You are given a location and are asked to return the weather of that location."
)

In [9]:
result = agent.invoke({
    "messages":[
        HumanMessage(content="Weather in NCR")
    ]
})
print(result)

{'messages': [HumanMessage(content='Weather in NCR', additional_kwargs={}, response_metadata={}, id='84d964fe-b743-4294-b057-bece8bcacf76'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for the weather in "NCR". This likely refers to the National Capital Region (NCR), which could be Delhi NCR (National Capital Region of India) or possibly New York City (NCR could stand for New York City). However, given the context of weather queries, it\'s most commonly referring to the National Capital Region of India (Delhi, Mumbai, etc.).\n\nLet me use the get_weather function to fetch the weather for "NCR". I\'ll need to specify the location parameter. Since "NCR" is a common abbreviation for the National Capital Region in India, I\'ll try that first. If it doesn\'t return useful results, I might need to clarify or try a different interpretation.\n\nActually, looking at the function, it requires a "location" parameter as a string. I should try "NCR" as the loca

#### Analyzing the output:

**Complete Response:**

```python
{'messages': [HumanMessage(content='Weather in NCR', additional_kwargs={}, response_metadata={}, id='ae3b5bda-b573-4f71-abd8-9694ed0aa940'), AIMessage(content="I'll get the weather for you! The NCR (National Capital Region) typically includes areas like Delhi, Gurugram, Noida, and Faridabad.\n\n", additional_kwargs={'reasoning_content': 'The user asked for the weather in "NCR", which is an acronym for the National Capital Region of India (encompassing Delhi, Gurugram, Noida, etc.).\n\nI don\'t have enough context to know exactly which city within the NCR they mean, but NCR as a region is commonly associated with Delhi. Since the get_weather tool takes a location string, I should call it with the location "NCR" or possibly "New Delhi, NCR".\n\nLet me call the get_weather function with the location NCR to retrieve the weather.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user asked for the weather in "NCR", which is an acronym for the National Capital Region of India (encompassing Delhi, Gurugram, Noida, etc.).\n\nI don\'t have enough context to know exactly which city within the NCR they mean, but NCR as a region is commonly associated with Delhi. Since the get_weather tool takes a location string, I should call it with the location "NCR" or possibly "New Delhi, NCR".\n\nLet me call the get_weather function with the location NCR to retrieve the weather.'}]}, response_metadata={'model_name': 'recursive/flint-v1.0:free', 'id': 'gen-1794312420-Tzld41jT98hXD2nNsDE4', 'created': 1794312420, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a1c55a-bc77-73f8-98e4-7897a88231e6-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'NCR'}, 'id': 'call_0b3b92b8cf4840b9aae2eead', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 150, 'total_tokens': 306, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 124}}), ToolMessage(content='The weather is 20 Celcius', name='get_weather', id='36a3050f-e189-41b7-b4d1-352a0b99ed07', tool_call_id='call_0b3b92b8cf4840b9aae2eead'), AIMessage(content='The current weather in NCR (National Capital Region) is **20°C**.', additional_kwargs={'reasoning_content': 'The tool returned "The weather is 20 Celcius" for NCR. I should present this clearly to the user.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The tool returned "The weather is 20 Celcius" for NCR. I should present this clearly to the user.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3.5-super-vl:free', 'id': 'gen-1794312422-7SQKVbFKV7m3qByLYbRI', 'created': 1794312422, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a1c55a-c4a7-75df-8091-98de24e777c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 496, 'output_tokens': 45, 'total_tokens': 541, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 26}})]}
```

**result["messages"]**

```python
[HumanMessage(content='Weather in NCR', additional_kwargs={}, response_metadata={}, id='ae3b5bda-b573-4f71-abd8-9694ed0aa940'),
 
 AIMessage(content="I'll get the weather for you! The NCR (National Capital Region) typically includes areas like Delhi, Gurugram, Noida, and Faridabad.\n\n", additional_kwargs={'reasoning_content': 'The user asked for the weather in "NCR", which is an acronym for the National Capital Region of India (encompassing Delhi, Gurugram, Noida, etc.).\n\nI don\'t have enough context to know exactly which city within the NCR they mean, but NCR as a region is commonly associated with Delhi. Since the get_weather tool takes a location string, I should call it with the location "NCR" or possibly "New Delhi, NCR".\n\nLet me call the get_weather function with the location NCR to retrieve the weather.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user asked for the weather in "NCR", which is an acronym for the National Capital Region of India (encompassing Delhi, Gurugram, Noida, etc.).\n\nI don\'t have enough context to know exactly which city within the NCR they mean, but NCR as a region is commonly associated with Delhi. Since the get_weather tool takes a location string, I should call it with the location "NCR" or possibly "New Delhi, NCR".\n\nLet me call the get_weather function with the location NCR to retrieve the weather.'}]}, response_metadata={'model_name': 'recursive/flint-v1.0:free', 'id': 'gen-1794312420-Tzld41jT98hXD2nNsDE4', 'created': 1794312420, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a1c55a-bc77-73f8-98e4-7897a88231e6-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'NCR'}, 'id': 'call_0b3b92b8cf4840b9aae2eead', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 150, 'total_tokens': 306, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 124}}),
 
 ToolMessage(content='The weather is 20 Celcius', name='get_weather', id='36a3050f-e189-41b7-b4d1-352a0b99ed07', tool_call_id='call_0b3b92b8cf4840b9aae2eead'),
 
 AIMessage(content='The current weather in NCR (National Capital Region) is **20°C**.', additional_kwargs={'reasoning_content': 'The tool returned "The weather is 20 Celcius" for NCR. I should present this clearly to the user.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The tool returned "The weather is 20 Celcius" for NCR. I should present this clearly to the user.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3.5-super-vl:free', 'id': 'gen-1794312422-7SQKVbFKV7m3qByLYbRI', 'created': 1794312422, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a1c55a-c4a7-75df-8091-98de24e777c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 496, 'output_tokens': 45, 'total_tokens': 541, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 26}})]
```
- We see the `HumanMessage` - we sent this.
- We see the `AIMessage` - AI reasoning output. 
  - We see that a tool call is requested as `tool_calls` is not empty.
- `ToolMessage` object denotes that the tool call was made and the output is returned.
  - the `tool_call_id` is same as in previous message to denote that this tool response is for this requested tool call.
- Another `AIMessage` denoting final answer.

**Lang chain has abstracted away the whole process of agentic loop and tool calling, the only thing we had to do was create the agent and provide tools, system prompt and messages**

In [10]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Weather in NCR
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_7a20ad3cf5194303b73d5aa1)
 Call ID: call_7a20ad3cf5194303b73d5aa1
  Args:
    location: NCR
================================= Tool Message =================================
Name: get_weather

The weather is 20 Celcius
================================== Ai Message ==================================



The weather in NCR is 20°C.


### Langchain Middleware:

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

It can intercept and modify the behaviour of the agent during the execution (before / after model call or tool call). It provides hooks for these.

![Image](https://mintcdn.com/langchain-5e9cc07a/RAP6mjwE5G00xYsA/oss/images/middleware_final.png?w=1650&fit=max&auto=format&n=RAP6mjwE5G00xYsA&q=85&s=3bca5ebf8aa56632b8a9826f7f112e57)

Some famous prebuilt middlewares :-
- ModelRetry
- Human-In-The-Loop
- Summarization
- ModelCallLimit
- ToolCallLimit

---

- https://docs.langchain.com/oss/python/langchain/middleware/overview
- https://reference.langchain.com/python/langchain/middleware

In [28]:
def build_hitl_middleware() -> HumanInTheLoopMiddleware:
    return HumanInTheLoopMiddleware(
        interrupt_on={
            "read_file": False,
            "list_files": False,
            "read_file_lines": False,
            "search_in_file": False,
            "write_file": {
                "allowed_decisions": ["approve", "edit", "reject"],
                "description": "Write or overwrite a file on disk",
            },
            "edit_file": {
                "allowed_decisions": ["approve", "edit", "reject"],
                "description": "Edit a file on disk",
            }
        },
        description_prefix="Coding agent needs your approval to move ahead"
    )

- Created a function for prebuilt middleware

In [34]:
def AuditMiddleware(AgentMiddleWare):
    """Append a json record after each tool call"""
    def wrap_tool_call(
        self, 
        request: ToolCallRequest, 
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        # make the tool call first
        result = handler(request)
        
        preview = ""
    
        if isinstance(result, ToolMessage):
            preview = str(result.content)[:200]
            self._write({
                "preview": preview,
                "tool": request.tool_call.get("name"),
                "arguments": request.tool_call.get("args"),
            })
        
        return result

    def _write(
        self,
        entry: dict[str, any],
    ) -> None:
        log_path = PROJECT_ROOT / ".audit.log"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open("a", encoding='utf-8') as f:
            f.write(json.dumps(entry) + "\n")

- Created a custom middleware by extending the AgentMiddleWare class
- We override its method as shown above to intercept at different points in agentic loop.

### Structured Outputs in Langchain:

- Using pydantic we can define the response strucutre.
- You define a class which extends the BaseModel from pydantic and define the fields there as we did before.

In [39]:
class CodingRequst(BaseModel):
    """Structured Representation of Coding Request"""
    task_type: Literal[
        "implement", 
        "debug", 
        "review", 
        "explain",
    ] = Field(description="type of taskt to be performed")

    objective: str = Field(description="A concise description of what needs to be achieved.")
    need_code_changes: bool = Field(description="Whether fulfilling this task requires modifying the source code")
    target_files: list[str] = Field(description="List of additional files related to the task")

In [40]:
structured_output_model = llm.with_structured_output(CodingRequst)

- `with_structured_output` allows langchain to get a model which will return the model which understands the output structure required.
- https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel/with_structured_output

Key considerations for structured output
- Method parameter: Some providers support different methods for structured output:
  - 'json_schema': Uses dedicated structured output features offered by the provider.
  - 'function_calling': Derives structured output by forcing a tool call that follows the given schema.
  - 'json_mode': A precursor to 'json_schema' offered by some providers. Generates valid JSON, but the schema must be described in the prompt.
- Include raw: Set include_raw=True to get both the parsed output and the raw AI message.
- Validation: Pydantic models provide automatic validation. TypedDict and JSON Schema require manual validation.
- Refusals: If the model refuses, structured output raises ModelRefusalError. See Handle a model refusal and Structured output.

See your provider’s integration page for supported methods and configuration options.

```python
from pydantic import BaseModel

class AnswerWithJustification(BaseModel):
    '''An answer to the user question along with justification for the answer.'''

    answer: str
    justification: str

model = ChatModel(model="model-name", temperature=0)
structured_model = model.with_structured_output(AnswerWithJustification)

structured_model.invoke(
    "What weighs more a pound of bricks or a pound of feathers"
)

# -> AnswerWithJustification(
#     answer='They weigh the same',
#     justification='Both a pound of bricks and a pound of feathers weigh one pound. The weight is the same, but the volume or density of the objects may differ.'
# )
```

----
- There is also a property on `create_agent` which allows passing in valid output formats.
```python
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5.5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
```
- https://docs.langchain.com/oss/python/langchain/structured-output

The docs specify strategy for structured outputs:

Use `response_format` to control how the agent returns structured data:

- ToolStrategy[StructuredResponseT]: Uses tool calling for structured output
- ProviderStrategy[StructuredResponseT]: Uses provider-native structured output
- type[StructuredResponseT]: Schema type - automatically selects best strategy based on model capabilities
- None: Structured output not explicitly requested

When a schema type is provided directly, LangChain automatically chooses:
ProviderStrategy if the model and provider chosen supports native structured output (e.g. OpenAI, Anthropic (Claude), or xAI (Grok)).
ToolStrategy for all other models.

Tools Strategy example :-
```python
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])
```

## [Short Term Memory](https://docs.langchain.com/oss/python/concepts/memory):

- Memory is basically every single interaction that has taken place between you and the agent.
- Currently we have been maintaining the memory manually - the `messages` array.
  - In each call we are passing the whole messages array.
  - Serialization / Deserialzation takes places which increase latency as the array increases.
  - When context increases we summarize it. (or use summarization middleware of langchain)
- Short-term memory, or thread-scoped memory, tracks the ongoing conversation by maintaining message history within a session. LangGraph manages short-term memory as a part of your agent’s state. State is persisted to a database using a checkpointer so the thread can be resumed at any time. Short-term memory updates when the graph is invoked or a step is completed, and the State is read at the start of each step.
- Long-term memory stores user-specific or application-level data across sessions and is shared across conversational threads. It can be recalled at any time and in any thread. Memories are scoped to any custom namespace, not just within a single thread ID. LangGraph provides stores (reference doc) to let you save and recall long-term memories.

![Image](https://mintcdn.com/langchain-5e9cc07a/dL5Sn6Cmy9pwtY0V/oss/images/short-vs-long.png?w=1650&fit=max&auto=format&n=dL5Sn6Cmy9pwtY0V&q=85&s=1997d5223d23ada411cce7d1170d6795)

- Langchain internally uses Langgraph to maintain short term memory.
  - It uses state object to maintain this.
  - A checkpointer is used to store the current context in the DB (Redis or any other).
    - It is like a middleware.
    - It is updated when the interaction happens / graph is invoked.
    - https://docs.langchain.com/oss/python/langgraph/add-memory#add-short-term-memory

```python
from langgraph.checkpoint.memory import InMemorySaver  
from langgraph.graph import StateGraph

checkpointer = InMemorySaver()

builder = StateGraph(...)
graph = builder.compile(checkpointer=checkpointer)

graph.invoke(
    {"messages": [{"role": "user", "content": "hi! i am Bob"}]},
    {"configurable": {"thread_id": "1"}},
)
```
- We create a graph with a checkpointer.
  - Then we pass in a configurable object with `thread_id` - every invoke must have this to use the same checkpointed context used previously.
- for production use a Redis / Postgres backed memory.

In [42]:
check_pointer = InMemorySaver()
def thread_config(thread_id: str) -> dict:
    return {
        "configurable": {
            "thread_id": "1",
        }
    }

In [54]:
agent_with_checkpointer = create_agent(
    model=llm,
    checkpointer=check_pointer
)

In [55]:
sample_thread_id = "1"

In [57]:
print(agent_with_checkpointer.invoke({"messages": HumanMessage("What is 2+2")}, thread_config(sample_thread_id))["messages"][-1].content)

2 + 2 = 4


In [58]:
print(agent_with_checkpointer.invoke({"messages": HumanMessage("What was the previous question")}, thread_config(sample_thread_id))["messages"][-1].content)

The previous question was **"What is 2+2"**.


- As we can see in previous example, even though we only sent a single message in the previous message, it still remembered our first question of `What is 2 + 2`.
- This was possible because the thread_id was same in the next call as well.
- It is to be noted that the context usage is still the same. For inference the whole memory is sent again. Only serialization / deserialization and network cost is avoided from this (which can be large if context window increases over time)